## Fetch sea level data from the Stormglass API

In [47]:
import requests

import arrow
import dotenv
import pandas as pd

from loguru import logger


dotenv.load_dotenv()


def request_sea_level_point(
    longitude: float, latitude: float, start_date: str, end_date: str
) -> pd.DataFrame:
    """
    Requests sea level from the Stormglass API.

    Parameters:
        longitude: Longitude in decimal degrees
        latitude: Latitude in decimal degrees
        start_date: Start date in format YYYYMMDD
        end_date: End date in format YYYYMMDD
    Returns:
        DataFrame with results
    """
    assert longitude > -180 and longitude < 180, (
        f"longitude must be between -180 and 180, got: {longitude}"
    )
    assert latitude > -90 and latitude < 90, (
        f"latitude must be between -90 and 90, got: {latitude}"
    )
    assert "STORMGLASS_API_KEY" in dotenv.dotenv_values(), (
        "missing .env value: 'STORMGLASS_API_KEY'"
    )

    start_time: arrow.Arrow = arrow.get(start_date, "YYYYMMDD")
    end_time: arrow.Arrow = arrow.get(end_date, "YYYYMMDD")

    response: requests.Response = requests.get(
        "https://api.stormglass.io/v2/tide/sea-level/point",
        params={
            "lng": longitude,
            "lat": latitude,
            "start": start_time.to(
                "UTC"
            ).timestamp(),  # Convert to UTC timestamp
            "end": end_time.to("UTC").timestamp(),  # Convert to UTC timestam
        },
        headers={
            "Authorization": dotenv.dotenv_values().get("STORMGLASS_API_KEY")
        },
    )

    response.raise_for_status()
    data: dict = response.json()

    # Parse data
    df: pd.DataFrame = _convert_response_to_data_frame(data)
    df["sea_level"] = df["sg"]
    df["datetime"] = pd.to_datetime(df["time"])
    df = df.drop(columns=["sg", "time"])

    return df


def _format_sealevel_response_metadata(metadata: dict) -> dict:
    """Formats the metadata of a Stormglass response."""
    station: dict = metadata.get("station")
    station_metadata: dict = {
        "longitude": station.get("lng"),
        "latitude": station.get("lat"),
        "distance": station.get("distance"),
        "name": station.get("name"),
        "source": station.get("source"),
    }
    request_metadata: dict = {
        "longitude": metadata.get("lng"),
        "latitude": metadata.get("lat"),
        "datum": metadata.get("datum"),
        "start": metadata.get("start"),
        "end": metadata.get("end"),
    }

    merged: dict = dict()
    for key, value in request_metadata.items():
        merged[f"request_{key}"] = value

    for key, value in station_metadata.items():
        merged[f"station_{key}"] = value

    return merged


def _convert_response_to_data_frame(response: dict) -> pd.DataFrame:
    """Converts a Stormglass reponse to a data frame."""
    df: pd.DataFrame = pd.DataFrame(response.get("data"))
    metadata: dict = _format_sealevel_response_metadata(response.get("meta"))
    for key, value in metadata.items():
        df[key] = value
    return df


# NOTE: Elephant rocks, Tasmania
longitude: float = 148.34169
latitude: float = -41.25289

start_date: str = "20090101"
end_date: str = "20211231"

start_time: arrow.Arrow = arrow.get(start_date, "YYYYMMDD")
end_time: arrow.Arrow = arrow.get(end_date, "YYYYMMDD")

df: pd.DataFrame = request_sea_level_point(
    longitude=longitude,
    latitude=latitude,
    start_date=start_date,
    end_date=end_date,
)

logger.info(df)

2025-09-11 14:15:01.005 | INFO     | __main__:<module>:109 -      request_longitude  request_latitude request_datum     request_start  \
0            148.34169         -41.25289           MSL  2009-01-01 00:00   
1            148.34169         -41.25289           MSL  2009-01-01 00:00   
2            148.34169         -41.25289           MSL  2009-01-01 00:00   
3            148.34169         -41.25289           MSL  2009-01-01 00:00   
4            148.34169         -41.25289           MSL  2009-01-01 00:00   
..                 ...               ...           ...               ...   
236          148.34169         -41.25289           MSL  2009-01-01 00:00   
237          148.34169         -41.25289           MSL  2009-01-01 00:00   
238          148.34169         -41.25289           MSL  2009-01-01 00:00   
239          148.34169         -41.25289           MSL  2009-01-01 00:00   
240          148.34169         -41.25289           MSL  2009-01-01 00:00   

          request_end  sta